# 01 – Explore Sources (VietDist)
Goal: fill in the **Initial File Exploration** table: ID, File, Format, Grain, Holds and Main keys.

## Step 1 – Setup

In [4]:
import pandas as pd
from pathlib import Path

DATA = Path("../data/raw")   # the notebook lives in notebooks/
DOCS = Path("../docs")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

assert DATA.exists(), f"Cannot find {DATA.resolve()}. Check where the notebook and the data folder are."

## Step 2 – Check all 10 files name, format, and size

In [5]:
files = sorted(f for f in DATA.iterdir() if f.suffix.lower() in (".csv", ".xlsx"))
inventory = pd.DataFrame({
    "file": [f.name for f in files],
    "format": [f.suffix[1:].upper() for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 2) for f in files],
})
print(len(files), "files, total", inventory.size_mb.sum().round(1), "MB")
inventory

10 files, total 25.3 MB


,file,format,size_mb
0,SRC01_sales_transactions.csv,CSV,20.62
1,SRC02_sales_target_plan.xlsx,XLSX,0.11
2,SRC03_customer_master.csv,CSV,0.38
3,SRC04_product_master.xlsx,XLSX,0.01
4,SRC05_distributor_orders.xlsx,XLSX,3.53
5,SRC06_distributor_master.csv,CSV,0.03
6,SRC07_employee_master.xlsx,XLSX,0.03
7,SRC08_territory_mapping.xlsx,XLSX,0.07
8,SRC09_return_transactions.csv,CSV,0.49
9,SRC10_promotion_program.xlsx,XLSX,0.01


## Step 3 – Helper functions

### Overall goals:
- Avoid repetition: write each check once, reuse it on all 10 files.
- Consistency: every file is profiled the same way, so results compare.

In [6]:
def load_source(name):
    """CSV → {"data": df}. Excel → {sheet_name: df} for every sheet."""
    # Goal: read any file into the SAME shape,
    #       code can always loop without caring whether the source was CSV or Excel.
    path = DATA / name
    if path.suffix.lower() == ".csv":
        return {"data": pd.read_csv(path)}      # wrap the single table in a dict
    return pd.read_excel(path, sheet_name=None)  


def overview(df, label=""):
    """Shape, first 5 rows, then dtype / nulls / unique count per column."""
    # Goal: a quick snapshot of a table to describe what it HOLDS, 
    #       and to spot obvious problems (wrong types, missing data, possible keys).
    print(f"=== {label}  shape={df.shape}")    # how many rows / columns
    display(df.head())                           # what real rows look like
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),             # wrong types (e.g. phone as int)
        "nulls": df.isna().sum(),                   # count of missing values
        "null_%": (df.isna().mean() * 100).round(1),# share of missing values
        "unique": df.nunique(),                     # = row count → possible key
        "example": df.iloc[0] if len(df) else None, # one sample value per column
    }))


def candidate_keys(df):
    """Single columns that are unique and never null, so could be a PK on their own."""
    # Goal: shortlist PK candidates automatically.
    return [c for c in df.columns if df[c].notna().all() and df[c].is_unique]


def id_columns(df):
    """Columns whose names look like keys (ending in _id, _code or id)."""
    # Goal: find likely FOREIGN keys by name. 
    return [c for c in df.columns
            if c.lower().endswith(("_id", "_code", "id")) or "_id_" in c.lower()]


def is_unique(df, cols):
    """True if this column combination has no duplicate rows."""
    # Goal: test a COMPOSITE key.
    return not df.duplicated(subset=cols).any()


def show_duplicates(df, col, n=1):
    """Show the rows for the most repeated value of `col`"""
    # Goal: help me DISCOVER the grain. If a column repeats, look at the
    #       duplicate rows and find which columns change between them –
    #       that column is the one to add to the key.
    vc = df[col].value_counts()
    vc = vc[vc > 1]                              # keep only repeated values
    if vc.empty:
        print(f"{col}: no repeated values")
        return
    print(f"{col}: {len(vc)} values repeat, the most frequent {vc.iloc[0]} times")
    for v in vc.index[:n]:                       # look at the top n repeated values
        sub = df[df[col] == v]
        display(sub)
        # dropna=False → a null vs a value also counts as "different"
        varying = [c for c in sub.columns if sub[c].nunique(dropna=False) > 1]
        print("→ Columns that differ between the duplicate rows:", varying)


def check_grain(df, cols):
    """Check whether the selected columns form a valid key in this dataset."""
    if not cols:
        print("GRAIN is empty. Review the data and specify the key columns.")
        return

    missing = [col for col in cols if col not in df.columns]
    if missing:
        print(f"Columns not found: {missing}")
        return

    unique = is_unique(df, cols)
    null_rows = int(df[cols].isna().any(axis=1).sum())

    print(f"GRAIN {cols}: unique={unique}, rows with a null key={null_rows}")

    if unique and null_rows == 0:
        print("The selected columns form a valid key in this dataset.")
        return

    if not unique:
        print("Duplicate keys found. Use show_duplicates() to inspect the affected rows.")

    if null_rows > 0:
        print("Null keys found. Inspect the affected rows before confirming the key.")

# Goal: keep results in memory for later steps.
TABLES = {}   # "SRC01" → {sheet: df}; Step 5 compares keys ACROSS tables with it
NOTES = {}    # my conclusions per file; Step 6 turns them into the markdown table

## Step 4 – Profile each file

For each file, work in this order:
1. **Load**: see which sheets the file has.
2. **Overview**: to write the **Holds** column.
3. **Candidate keys**: see which columns are unique alone and which look like keys.
4. **Test the grain**: edit `GRAIN = [...]` until get ✅. That gives **Grain** and the **PK**.
5. **Notes**: fill in `NOTES`.



### SRC01 – `SRC01_sales_transactions.csv`


In [7]:
TABLES["SRC01"] = load_source("SRC01_sales_transactions.csv")
src01 = TABLES["SRC01"]
print("Sheets:", list(src01.keys()))
for sheet, df in src01.items():
    overview(df, f"SRC01 / {sheet}")

Sheets: ['data']
=== SRC01 / data  shape=(119101, 21)


,order_id,order_date,order_month,order_quarter,order_year,customer_id,region,province,channel,employee_id,product_id,product_category,quantity,unit_price,discount_pct,discount_amount,gross_amount,net_amount,delivery_status,payment_method,payment_status
0,ORD0010001,2024-08-22,8,3,2024,CUS01590,Miền Nam,TP.HCM,Modern Trade,EMP1079,PRD0055,Thực phẩm,109,8500,2,18500.0,926500,908000.0,Delivered,Công nợ,Paid
1,ORD0010002,2024-12-28,12,4,2024,CUS01166,Miền Nam,Vũng Tàu,E-commerce,EMP1092,PRD0057,Thực phẩm,59,50000,0,0.0,2950000,2950000.0,Delivered,Công nợ,Paid
2,ORD0010002,2024-12-28,12,4,2024,CUS01166,Miền Nam,Vũng Tàu,E-commerce,EMP1092,PRD0036,Thực phẩm,9,30000,0,0.0,270000,270000.0,Delivering,Tiền mặt,Paid
3,ORD0010002,2024-12-28,12,4,2024,CUS01166,Miền Nam,Vũng Tàu,E-commerce,EMP1092,PRD0087,Gia dụng,7,15000,0,0.0,105000,105000.0,Delivering,Công nợ,Paid
4,ORD0010003,2024-03-19,3,1,2024,CUS01832,Miền Trung,Quảng Ngãi,Modern Trade,EMP1047,PRD0069,Chăm sóc cá nhân,34,40000,0,0.0,1360000,1360000.0,Delivered,Công nợ,Unpaid


,dtype,nulls,null_%,unique,example
order_id,object,0,0.0,50000,ORD0010001
order_date,object,0,0.0,366,2024-08-22
order_month,int64,0,0.0,12,8
order_quarter,int64,0,0.0,4,3
order_year,int64,0,0.0,1,2024
customer_id,object,0,0.0,1802,CUS01590
region,object,0,0.0,3,Miền Nam
province,object,0,0.0,23,TP.HCM
channel,object,0,0.0,3,Modern Trade
employee_id,object,0,0.0,108,EMP1079


In [8]:
SHEET = "data"
df = src01[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: []
Look like keys     : ['order_id', 'customer_id', 'employee_id', 'product_id']


In [9]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [10]:
NOTES["SRC01"] = {
    "file": "SRC01_sales_transactions.csv",
    "format": "CSV",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC02 – `SRC02_sales_target_plan.xlsx`

💡 **Hint:** Several sheets. Which hold data and which are summaries? There are two plan versions. Is the version part of the key?

In [11]:
TABLES["SRC02"] = load_source("SRC02_sales_target_plan.xlsx")
src02 = TABLES["SRC02"]
print("Sheets:", list(src02.keys()))
for sheet, df in src02.items():
    overview(df, f"SRC02 / {sheet}")

Sheets: ['Plan_v1_Original', 'Plan_v2_Adjustment_H2', 'Summary']
=== SRC02 / Plan_v1_Original  shape=(1332, 13)


,plan_version,version_date,effective_from,effective_to,employee_id,employee_name,region,team,year,month,target_revenue,target_quantity,target_new_customers
0,v1,2024-01-05,2024-01-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,1,359500000,19972,6
1,v1,2024-01-05,2024-01-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,2,179800000,9988,10
2,v1,2024-01-05,2024-01-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,3,239700000,13316,5
3,v1,2024-01-05,2024-01-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,4,209700000,11650,12
4,v1,2024-01-05,2024-01-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,5,239700000,13316,8


,dtype,nulls,null_%,unique,example
plan_version,object,0,0.0,1,v1
version_date,object,0,0.0,1,2024-01-05
effective_from,object,0,0.0,1,2024-01-01
effective_to,object,0,0.0,1,2024-12-31
employee_id,object,0,0.0,111,EMP1001
employee_name,object,0,0.0,109,Hồ Thị Bình
region,object,0,0.0,3,Miền Bắc
team,object,0,0.0,9,Team Bắc 1
year,int64,0,0.0,1,2024
month,int64,0,0.0,12,1


=== SRC02 / Plan_v2_Adjustment_H2  shape=(618, 13)


,plan_version,version_date,effective_from,effective_to,employee_id,employee_name,region,team,year,month,target_revenue,target_quantity,target_new_customers
0,v2,2024-07-03,2024-07-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,7,224300000,12461,8
1,v2,2024-07-03,2024-07-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,8,256400000,14244,11
2,v2,2024-07-03,2024-07-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,9,288400000,16022,5
3,v2,2024-07-03,2024-07-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,10,224300000,12461,4
4,v2,2024-07-03,2024-07-01,2024-12-31,EMP1001,Hồ Thị Bình,Miền Bắc,Team Bắc 1,2024,11,256400000,14244,5


,dtype,nulls,null_%,unique,example
plan_version,object,0,0.0,1,v2
version_date,object,0,0.0,1,2024-07-03
effective_from,object,0,0.0,1,2024-07-01
effective_to,object,0,0.0,1,2024-12-31
employee_id,object,0,0.0,103,EMP1001
employee_name,object,0,0.0,101,Hồ Thị Bình
region,object,0,0.0,3,Miền Bắc
team,object,0,0.0,9,Team Bắc 1
year,int64,0,0.0,1,2024
month,int64,0,0.0,6,7


=== SRC02 / Summary  shape=(5, 3)


,Thông tin,Plan v1,Plan v2
0,Phiên bản,v1,v2
1,Ngày hiệu lực,01/01/2024,01/07/2024
2,Số nhân viên,111,103
3,Tổng target cả năm (VNĐ),"279,998,400,000","279,996,800,000"
4,Ghi chú,Kế hoạch đầu năm 12 tháng,"8 NV nghỉ, 5 NV chuyển vùng"


,dtype,nulls,null_%,unique,example
Thông tin,object,0,0.0,5,Phiên bản
Plan v1,object,0,0.0,5,v1
Plan v2,object,0,0.0,5,v2


In [12]:
SHEET = list(src02.keys())[0]   # TODO: change to the data sheet you want to profile
df = src02[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: []
Look like keys     : ['employee_id']


In [13]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [14]:
NOTES["SRC02"] = {
    "file": "SRC02_sales_target_plan.xlsx",
    "format": "XLSX",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC03 – `SRC03_customer_master.csv`

💡 **Hint:** A simple master table, so start here. Watch the data type of the phone column and the null rate of each column.

In [15]:
TABLES["SRC03"] = load_source("SRC03_customer_master.csv")
src03 = TABLES["SRC03"]
print("Sheets:", list(src03.keys()))
for sheet, df in src03.items():
    overview(df, f"SRC03 / {sheet}")

Sheets: ['data']
=== SRC03 / data  shape=(2000, 12)


,customer_id,customer_name,customer_type,channel,province,region,address,phone,tax_code,join_date,credit_limit,status
0,CUS00001,Cửa hàng tiện lợi Hồ Đức Cường,Cửa hàng tiện lợi,Modern Trade,Bình Dương,Miền Nam,"Số 127 đường Điện Biên Phủ, Bình Dương",2868978194,NaN,2023-11-17,100000000,Active
1,CUS00002,Canteen Dương Quốc Dũng,Canteen,Traditional Trade,Quảng Ngãi,Miền Trung,"Số 405 đường Điện Biên Phủ, Quảng Ngãi",2618846539,2.416011e+09,2022-05-03,200000000,Active
2,CUS00003,Nhà hàng Trần Công Linh,Nhà hàng,Traditional Trade,Đồng Nai,Miền Nam,"Số 470 đường Trần Phú, Đồng Nai",2598054173,2.683447e+09,2018-12-14,200000000,Active
3,CUS00004,Tạp hóa Hồ Hồng Hà,Tạp hóa,Traditional Trade,Tiền Giang,Miền Nam,"Số 331 đường Điện Biên Phủ, Tiền Giang",2778248169,NaN,2022-01-22,50000000,Active
4,CUS00005,Nhà hàng Dương Hữu Nam,Nhà hàng,Traditional Trade,Bắc Ninh,Miền Bắc,"Số 116 đường Nguyễn Huệ, Bắc Ninh",2320356922,4.384950e+09,2023-09-11,50000000,Active


,dtype,nulls,null_%,unique,example
customer_id,object,0,0.0,2000,CUS00001
customer_name,object,0,0.0,1942,Cửa hàng tiện lợi Hồ Đức Cường
customer_type,object,0,0.0,6,Cửa hàng tiện lợi
channel,object,0,0.0,3,Modern Trade
province,object,0,0.0,23,Bình Dương
region,object,0,0.0,3,Miền Nam
address,object,0,0.0,1976,"Số 127 đường Điện Biên Phủ, Bình Dương"
phone,int64,0,0.0,2000,2868978194
tax_code,float64,1035,51.7,965,NaN
join_date,object,0,0.0,1291,2023-11-17


In [16]:
SHEET = "data"
df = src03[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: ['customer_id', 'phone']
Look like keys     : ['customer_id', 'tax_code']


In [17]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [18]:
NOTES["SRC03"] = {
    "file": "SRC03_customer_master.csv",
    "format": "CSV",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC04 – `SRC04_product_master.xlsx`

💡 **Hint:** Has a summary sheet to skip. Has every product been sold? (Check in Step 5.)

In [19]:
TABLES["SRC04"] = load_source("SRC04_product_master.xlsx")
src04 = TABLES["SRC04"]
print("Sheets:", list(src04.keys()))
for sheet, df in src04.items():
    overview(df, f"SRC04 / {sheet}")

Sheets: ['Products', 'Category_Summary']
=== SRC04 / Products  shape=(100, 10)


,product_id,product_name,category,sub_category,unit,unit_price,cost_price,weight_gram,status,launch_date
0,PRD0001,Nước giải khát Loại B,Thức uống,Nước giải khát,Hộp,12000,7700,250,Active,2023-06-09
1,PRD0002,Nước giải khát Đặc biệt,Thức uống,Nước giải khát,Chai,9000,6400,500,Active,2019-10-11
2,PRD0003,Nước giải khát Premium,Thức uống,Nước giải khát,Lon,15000,8300,300,Active,2019-04-10
3,PRD0004,Nước giải khát Standard,Thức uống,Nước giải khát,Chai,25000,16500,200,Active,2021-04-10
4,PRD0005,Nước uống đóng chai Loại B,Thức uống,Nước uống đóng chai,Hộp,50000,31000,300,Active,2020-09-25


,dtype,nulls,null_%,unique,example
product_id,object,0,0.0,100,PRD0001
product_name,object,0,0.0,100,Nước giải khát Loại B
category,object,0,0.0,4,Thức uống
sub_category,object,0,0.0,17,Nước giải khát
unit,object,0,0.0,6,Hộp
unit_price,int64,0,0.0,12,12000
cost_price,int64,0,0.0,77,7700
weight_gram,int64,0,0.0,7,250
status,object,0,0.0,2,Active
launch_date,object,0,0.0,100,2023-06-09


=== SRC04 / Category_Summary  shape=(4, 4)


,category,total_skus,active_skus,avg_price
0,Chăm sóc cá nhân,19,17,23105.263158
1,Gia dụng,15,15,23900.000000
2,Thức uống,31,29,25919.354839
3,Thực phẩm,35,31,19114.285714


,dtype,nulls,null_%,unique,example
category,object,0,0.0,4,Chăm sóc cá nhân
total_skus,int64,0,0.0,4,19
active_skus,int64,0,0.0,4,17
avg_price,float64,0,0.0,4,23105.263158


In [20]:
SHEET = list(src04.keys())[0]   # TODO: change to the data sheet you want to profile
df = src04[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: ['product_id', 'product_name', 'launch_date']
Look like keys     : ['product_id']


In [21]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [22]:
NOTES["SRC04"] = {
    "file": "SRC04_product_master.xlsx",
    "format": "XLSX",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC05 – `SRC05_distributor_orders.xlsx`

💡 **Hint:** Like SRC01, but for distributor orders. Is the grain the same as SRC01?

In [23]:
TABLES["SRC05"] = load_source("SRC05_distributor_orders.xlsx")
src05 = TABLES["SRC05"]
print("Sheets:", list(src05.keys()))
for sheet, df in src05.items():
    overview(df, f"SRC05 / {sheet}")

Sheets: ['Orders']
=== SRC05 / Orders  shape=(35945, 21)


,order_id,order_date,order_month,order_quarter,distributor_id,region,channel,product_id,product_category,qty_ordered,qty_delivered,fill_rate_pct,unit_price_list,distributor_price,gross_amount,delivered_amount,expected_delivery_date,actual_delivery_date,ontime_delivery,delivery_status,payment_terms
0,DORD0020001,2024-01-13,1,1,DIST0102,Miền Nam,Traditional Trade,PRD0054,Thực phẩm,226,200,88.50,10000,8980,2029480,1796000,2024-01-15,2024-01-17,No,Delivered,NET30
1,DORD0020001,2024-01-13,1,1,DIST0102,Miền Nam,Traditional Trade,PRD0047,Thực phẩm,49,45,91.84,20000,16210,794290,729450,2024-01-15,2024-01-16,No,Delivered,NET15
2,DORD0020001,2024-01-13,1,1,DIST0102,Miền Nam,Traditional Trade,PRD0092,Gia dụng,174,136,78.16,12000,10170,1769580,1383120,2024-01-15,2024-01-14,Yes,Delivered,NET45
3,DORD0020001,2024-01-13,1,1,DIST0102,Miền Nam,Traditional Trade,PRD0038,Thực phẩm,205,192,93.66,10000,8150,1670750,1564800,2024-01-15,2024-01-15,Yes,Partial,NET30
4,DORD0020001,2024-01-13,1,1,DIST0102,Miền Nam,Traditional Trade,PRD0064,Thực phẩm,237,229,96.62,20000,17900,4242300,4099100,2024-01-15,2024-01-18,No,Delivered,NET45


,dtype,nulls,null_%,unique,example
order_id,object,0,0.0,8000,DORD0020001
order_date,object,0,0.0,366,2024-01-13
order_month,int64,0,0.0,12,1
order_quarter,int64,0,0.0,4,1
distributor_id,object,0,0.0,125,DIST0102
region,object,0,0.0,3,Miền Nam
channel,object,0,0.0,3,Traditional Trade
product_id,object,0,0.0,92,PRD0054
product_category,object,0,0.0,4,Thực phẩm
qty_ordered,int64,0,0.0,481,226


In [24]:
SHEET = list(src05.keys())[0]   # TODO: change to the data sheet you want to profile
df = src05[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: []
Look like keys     : ['order_id', 'distributor_id', 'product_id']


In [25]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [26]:
NOTES["SRC05"] = {
    "file": "SRC05_distributor_orders.xlsx",
    "format": "XLSX",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC06 – `SRC06_distributor_master.csv`

💡 **Hint:** A master table. Which distributors have never ordered? (Check in Step 5.)

In [27]:
TABLES["SRC06"] = load_source("SRC06_distributor_master.csv")
src06 = TABLES["SRC06"]
print("Sheets:", list(src06.keys()))
for sheet, df in src06.items():
    overview(df, f"SRC06 / {sheet}")

Sheets: ['data']
=== SRC06 / data  shape=(138, 14)


,distributor_id,distributor_name,tier,channel,province,region,contact_person,phone,email,tax_code,join_date,credit_limit,status,assigned_supervisor_id
0,DIST0001,Công ty CP Trần Minh Hải Dịch vụ,Đại lý cấp 2,Traditional Trade,Hà Nội,Miền Bắc,Lê Hữu Ngân,2320109001,dist0001@gmail.com,1059726454,2021-02-18,1000000000,Active,EMP1097
1,DIST0002,Công ty CP Lê Công Mai Thương mại,Điểm bán lẻ,Traditional Trade,Hà Nội,Miền Bắc,Dương Hữu Long,2879902880,dist0002@gmail.com,5119178016,2022-03-17,1000000000,Active,EMP1067
2,DIST0003,Công ty TNHH Nguyễn Công Khoa Thương mại,Điểm bán lẻ,E-commerce,Hà Nội,Miền Bắc,Phạm Thị Nam,2631546491,dist0003@gmail.com,5050102995,2015-09-27,500000000,Active,EMP1082
3,DIST0004,Công ty TNHH Nguyễn Hồng Thắng Phân phối,Đại lý cấp 2,Traditional Trade,Hà Nội,Miền Bắc,Đỗ Đức Nam,2268095397,dist0004@gmail.com,6846246925,2019-12-31,500000000,Active,EMP1012
4,DIST0005,Công ty TNHH Ngô Thị Hà Phân phối,Đại lý cấp 2,E-commerce,Hà Nội,Miền Bắc,Bùi Đức Thắng,2634814619,dist0005@gmail.com,6474777663,2019-04-20,2000000000,Active,EMP1043


,dtype,nulls,null_%,unique,example
distributor_id,object,0,0.0,138,DIST0001
distributor_name,object,0,0.0,138,Công ty CP Trần Minh Hải Dịch vụ
tier,object,0,0.0,3,Đại lý cấp 2
channel,object,0,0.0,3,Traditional Trade
province,object,0,0.0,23,Hà Nội
region,object,0,0.0,3,Miền Bắc
contact_person,object,0,0.0,136,Lê Hữu Ngân
phone,int64,0,0.0,138,2320109001
email,object,0,0.0,138,dist0001@gmail.com
tax_code,int64,0,0.0,138,1059726454


In [28]:
SHEET = "data"
df = src06[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: ['distributor_id', 'distributor_name', 'phone', 'email', 'tax_code']
Look like keys     : ['distributor_id', 'tax_code', 'assigned_supervisor_id']


In [29]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [30]:
NOTES["SRC06"] = {
    "file": "SRC06_distributor_master.csv",
    "format": "CSV",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC07 – `SRC07_employee_master.xlsx`

💡 **Hint:** Several sheets, each with its own grain. In the change-log sheet, what is one row?

In [31]:
TABLES["SRC07"] = load_source("SRC07_employee_master.xlsx")
src07 = TABLES["SRC07"]
print("Sheets:", list(src07.keys()))
for sheet, df in src07.items():
    overview(df, f"SRC07 / {sheet}")

Sheets: ['Employee_v1_StartYear', 'Employee_v2_Adjustment', 'Change_Log']
=== SRC07 / Employee_v1_StartYear  shape=(114, 15)


,employee_id,full_name,gender,date_of_birth,join_date,position,region,team,email,phone,status,version,effective_date,resign_date,transfer_note
0,EMP1001,Hồ Thị Bình,Nữ,1980-06-29,2017-07-03,Sales Supervisor,Miền Bắc,Team Bắc 1,emp1001@vietdist.com.vn,928728463,Active,v1,2024-01-01,NaN,NaN
1,EMP1002,Ngô Thị Hà,Nữ,1990-09-18,2018-05-03,Sales Representative,Miền Bắc,Team Bắc 1,emp1002@vietdist.com.vn,939345092,Active,v1,2024-01-01,NaN,NaN
2,EMP1003,Phạm Bá Bình,Nam,2002-03-23,2022-09-15,Sales Representative,Miền Bắc,Team Bắc 1,emp1003@vietdist.com.vn,970291817,Active,v1,2024-01-01,NaN,NaN
3,EMP1004,Đỗ Minh An,Nam,1999-06-25,2021-10-25,Senior Sales Representative,Miền Bắc,Team Bắc 1,emp1004@vietdist.com.vn,930868105,Active,v1,2024-01-01,NaN,NaN
4,EMP1005,Phạm Quốc Hùng,Nam,1998-07-10,2019-02-01,Senior Sales Representative,Miền Bắc,Team Bắc 1,emp1005@vietdist.com.vn,956164955,Active,v1,2024-01-01,NaN,NaN


,dtype,nulls,null_%,unique,example
employee_id,object,0,0.0,114,EMP1001
full_name,object,0,0.0,112,Hồ Thị Bình
gender,object,0,0.0,2,Nữ
date_of_birth,object,0,0.0,114,1980-06-29
join_date,object,0,0.0,112,2017-07-03
position,object,0,0.0,4,Sales Supervisor
region,object,0,0.0,3,Miền Bắc
team,object,0,0.0,10,Team Bắc 1
email,object,0,0.0,114,emp1001@vietdist.com.vn
phone,int64,0,0.0,114,928728463


=== SRC07 / Employee_v2_Adjustment  shape=(114, 15)


,employee_id,full_name,gender,date_of_birth,join_date,position,region,team,email,phone,status,version,effective_date,resign_date,transfer_note
0,EMP1001,Hồ Thị Bình,Nữ,1980-06-29,2017-07-03,Sales Supervisor,Miền Bắc,Team Bắc 1,emp1001@vietdist.com.vn,928728463,Active,v2,2024-07-01,NaN,NaN
1,EMP1002,Ngô Thị Hà,Nữ,1990-09-18,2018-05-03,Sales Representative,Miền Bắc,Team Bắc 1,emp1002@vietdist.com.vn,939345092,Active,v2,2024-07-01,NaN,NaN
2,EMP1003,Phạm Bá Bình,Nam,2002-03-23,2022-09-15,Sales Representative,Miền Bắc,Team Bắc 1,emp1003@vietdist.com.vn,970291817,Active,v2,2024-07-01,NaN,NaN
3,EMP1004,Đỗ Minh An,Nam,1999-06-25,2021-10-25,Senior Sales Representative,Miền Trung,Team Trung 2,emp1004@vietdist.com.vn,930868105,Active,v2,2024-07-01,NaN,Chuyển từ Miền Bắc sang Miền Trung
4,EMP1005,Phạm Quốc Hùng,Nam,1998-07-10,2019-02-01,Senior Sales Representative,Miền Bắc,Team Bắc 1,emp1005@vietdist.com.vn,956164955,Active,v2,2024-07-01,NaN,NaN


,dtype,nulls,null_%,unique,example
employee_id,object,0,0.0,114,EMP1001
full_name,object,0,0.0,112,Hồ Thị Bình
gender,object,0,0.0,2,Nữ
date_of_birth,object,0,0.0,114,1980-06-29
join_date,object,0,0.0,112,2017-07-03
position,object,0,0.0,4,Sales Supervisor
region,object,0,0.0,3,Miền Bắc
team,object,0,0.0,10,Team Bắc 1
email,object,0,0.0,114,emp1001@vietdist.com.vn
phone,int64,0,0.0,114,928728463


=== SRC07 / Change_Log  shape=(13, 8)


,employee_id,employee_name,change_type,effective_date,old_region,new_region,old_team,new_team
0,EMP1035,Đỗ Hữu Long,Nghỉ việc,2024-07-01,Miền Trung,NaN,Team Trung 1,NaN
1,EMP1054,Hoàng Văn Hùng,Nghỉ việc,2024-07-01,Miền Nam,NaN,Team Nam 1,NaN
2,EMP1075,Bùi Văn Lộc,Nghỉ việc,2024-07-01,Miền Nam,NaN,Team Nam 2,NaN
3,EMP1051,Lý Thanh Phúc,Nghỉ việc,2024-07-01,Miền Trung,NaN,Team Trung 2,NaN
4,EMP1070,Phạm Xuân Linh,Nghỉ việc,2024-07-01,Miền Nam,NaN,Team Nam 2,NaN


,dtype,nulls,null_%,unique,example
employee_id,object,0,0.0,13,EMP1035
employee_name,object,0,0.0,13,Đỗ Hữu Long
change_type,object,0,0.0,2,Nghỉ việc
effective_date,object,0,0.0,1,2024-07-01
old_region,object,0,0.0,3,Miền Trung
new_region,object,8,61.5,3,NaN
old_team,object,0,0.0,9,Team Trung 1
new_team,object,8,61.5,5,NaN


In [32]:
SHEET = list(src07.keys())[0]   # TODO: change to the data sheet you want to profile
df = src07[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: ['employee_id', 'date_of_birth', 'email', 'phone']
Look like keys     : ['employee_id']


In [33]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [34]:
NOTES["SRC07"] = {
    "file": "SRC07_employee_master.xlsx",
    "format": "XLSX",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC08 – `SRC08_territory_mapping.xlsx`

💡 **Hint:** One row describes a RELATIONSHIP between which two things? Is this table still correct after employees transfer regions?

In [35]:
TABLES["SRC08"] = load_source("SRC08_territory_mapping.xlsx")
src08 = TABLES["SRC08"]
print("Sheets:", list(src08.keys()))
for sheet, df in src08.items():
    overview(df, f"SRC08 / {sheet}")

Sheets: ['Sheet1']
=== SRC08 / Sheet1  shape=(1843, 8)


,territory_id,employee_id,customer_id,region,team,effective_date,expiry_date,version
0,TER00001,EMP1002,CUS01358,Miền Bắc,Team Bắc 1,2024-01-01,9999-12-31,v1
1,TER00002,EMP1002,CUS00714,Miền Bắc,Team Bắc 1,2024-01-01,9999-12-31,v1
2,TER00003,EMP1002,CUS00406,Miền Bắc,Team Bắc 1,2024-01-01,9999-12-31,v1
3,TER00004,EMP1002,CUS00400,Miền Bắc,Team Bắc 1,2024-01-01,9999-12-31,v1
4,TER00005,EMP1002,CUS01486,Miền Bắc,Team Bắc 1,2024-01-01,9999-12-31,v1


,dtype,nulls,null_%,unique,example
territory_id,object,0,0.0,1843,TER00001
employee_id,object,0,0.0,102,EMP1002
customer_id,object,0,0.0,1843,CUS01358
region,object,0,0.0,3,Miền Bắc
team,object,0,0.0,9,Team Bắc 1
effective_date,object,0,0.0,1,2024-01-01
expiry_date,object,0,0.0,1,9999-12-31
version,object,0,0.0,1,v1


In [36]:
SHEET = list(src08.keys())[0]   # TODO: change to the data sheet you want to profile
df = src08[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: ['territory_id', 'customer_id']
Look like keys     : ['territory_id', 'employee_id', 'customer_id']


In [37]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [38]:
NOTES["SRC08"] = {
    "file": "SRC08_territory_mapping.xlsx",
    "format": "XLSX",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC09 – `SRC09_return_transactions.csv`

💡 **Hint:** Which columns link a return back to its sale? Do the column names match SRC01?

In [39]:
TABLES["SRC09"] = load_source("SRC09_return_transactions.csv")
src09 = TABLES["SRC09"]
print("Sheets:", list(src09.keys()))
for sheet, df in src09.items():
    overview(df, f"SRC09 / {sheet}")

Sheets: ['data']
=== SRC09 / data  shape=(3665, 14)


,return_id,original_order_id,return_date,return_month,customer_id,employee_id,product_id,region,province,return_quantity,unit_price,return_amount,return_reason,status
0,RET000001,ORD0044684,2024-03-13,3,CUS00166,EMP1039,PRD0081,Miền Trung,Đà Nẵng,16,30000,480000,Hàng lỗi,Đã xử lý
1,RET000002,ORD0044962,2024-09-13,9,CUS01548,EMP1044,PRD0078,Miền Trung,Quảng Nam,21,10000,210000,Khách từ chối nhận,Từ chối
2,RET000003,ORD0029586,2024-07-31,7,CUS01590,EMP1079,PRD0012,Miền Nam,TP.HCM,3,12000,36000,Hàng hết hạn,Đã xử lý
3,RET000004,ORD0041144,2024-09-17,9,CUS01864,EMP1042,PRD0028,Miền Trung,Đà Nẵng,23,15000,345000,Hàng vỡ vận chuyển,Đã xử lý
4,RET000005,ORD0029299,2024-02-21,2,CUS01140,EMP1039,PRD0046,Miền Trung,Khánh Hòa,16,35000,560000,Khách từ chối nhận,Đã xử lý


,dtype,nulls,null_%,unique,example
return_id,object,0,0.0,3665,RET000001
original_order_id,object,0,0.0,3564,ORD0044684
return_date,object,0,0.0,362,2024-03-13
return_month,int64,0,0.0,12,3
customer_id,object,0,0.0,967,CUS00166
employee_id,object,0,0.0,106,EMP1039
product_id,object,0,0.0,92,PRD0081
region,object,0,0.0,3,Miền Trung
province,object,0,0.0,23,Đà Nẵng
return_quantity,int64,0,0.0,59,16


In [40]:
SHEET = "data"
df = src09[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: ['return_id']
Look like keys     : ['return_id', 'original_order_id', 'customer_id', 'employee_id', 'product_id']


In [41]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [42]:
NOTES["SRC09"] = {
    "file": "SRC09_return_transactions.csv",
    "format": "CSV",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

### SRC10 – `SRC10_promotion_program.xlsx`

💡 **Hint:** Print a few values of `applicable_products`. How many products are in one cell?

In [43]:
TABLES["SRC10"] = load_source("SRC10_promotion_program.xlsx")
src10 = TABLES["SRC10"]
print("Sheets:", list(src10.keys()))
for sheet, df in src10.items():
    overview(df, f"SRC10 / {sheet}")

Sheets: ['Sheet1']
=== SRC10 / Sheet1  shape=(40, 14)


,promotion_id,promotion_name,promotion_type,target_channel,target_region,start_date,end_date,applicable_products,discount_pct,min_order_quantity,budget_vnd,actual_cost_vnd,status,created_by
0,PROMO001,CT_KHAI_TRUONG_2024_01,Tặng hàng kèm,E-commerce,Miền Trung,2024-04-13,2024-05-19,PRD0017|PRD0040,8,0,278000000,296000000,Completed,Phạm Minh Việt
1,PROMO002,CT_XUAN_2024_02,Tích điểm đổi quà,Modern Trade,Miền Bắc,2024-09-07,2024-10-16,PRD0008|PRD0069|PRD0048|PRD0029|PRD0013|PRD0020,10,0,64000000,56000000,Completed,Lê Hồng Thảo
2,PROMO003,CT_KHAI_TRUONG_2024_03,Tích điểm đổi quà,E-commerce,All,2024-01-02,2024-01-28,PRD0014|PRD0064|PRD0037|PRD0079|PRD0029,12,10,445000000,414000000,Completed,Đặng Hồng Linh
3,PROMO004,CT_HE_2024_04,Chiết khấu theo sản lượng,All,Miền Nam,2024-05-03,2024-05-29,PRD0006|PRD0003|PRD0050|PRD0100|PRD0018,20,10,169000000,106000000,Completed,Lê Văn Thảo
4,PROMO005,CT_KHAI_TRUONG_2024_05,Giảm giá trực tiếp,E-commerce,All,2024-01-05,2024-01-24,PRD0083|PRD0007|PRD0003|PRD0075|PRD0006|PRD0048,10,0,74000000,53000000,Completed,Bùi Hồng Nam


,dtype,nulls,null_%,unique,example
promotion_id,object,0,0.0,40,PROMO001
promotion_name,object,0,0.0,40,CT_KHAI_TRUONG_2024_01
promotion_type,object,0,0.0,5,Tặng hàng kèm
target_channel,object,0,0.0,4,E-commerce
target_region,object,0,0.0,4,Miền Trung
start_date,object,0,0.0,38,2024-04-13
end_date,object,0,0.0,39,2024-05-19
applicable_products,object,0,0.0,40,PRD0017|PRD0040
discount_pct,int64,0,0.0,6,8
min_order_quantity,int64,0,0.0,4,0


In [44]:
SHEET = list(src10.keys())[0]   # TODO: change to the data sheet you want to profile
df = src10[SHEET]

print("Unique on their own:", candidate_keys(df))
print("Look like keys     :", id_columns(df))

Unique on their own: ['promotion_id', 'promotion_name', 'applicable_products', 'budget_vnd', 'created_by']
Look like keys     : ['promotion_id']


In [45]:
# Test the grain: edit the list and re-run until you get ✅
GRAIN = []   # TODO, e.g. ["order_id"] or ["order_id", "product_id"]
check_grain(df, GRAIN)

# If there are duplicates, uncomment the line below to see how they differ:
# show_duplicates(df, "order_id")

GRAIN is empty. Review the data and specify the key columns.


In [46]:
NOTES["SRC10"] = {
    "file": "SRC10_promotion_program.xlsx",
    "format": "XLSX",
    "grain": "",       # one row = ...
    "holds": "",       # what the table contains, row count, anything notable
    "keys": "",        # PK: ... ; FK: ...
    "issues": "",      # anything odd to log in data_issues
}

## Step 5 – Check foreign keys

The first cell **automatically finds** columns with the same name in two tables where one side is unique, i.e. a possible PK → FK pair.
- `orphans` > 0: values in the child table that don't exist in the parent.
- `unused` > 0: parent values never used (e.g. products never sold).

For columns with different names (e.g. `original_order_id` → `order_id`), check them yourself in the second cell.

In [47]:
flat = {f"{sid}/{sh}": d for sid, sheets_ in TABLES.items() for sh, d in sheets_.items()}
rows = []
for parent, pdf in flat.items():
    for col in candidate_keys(pdf):
        for child, cdf in flat.items():
            if child == parent or col not in cdf.columns:
                continue
            p, c = set(pdf[col].dropna()), set(cdf[col].dropna())
            rows.append({"parent (PK)": parent, "child (FK)": child, "column": col,
                         "orphans": len(c - p), "unused": len(p - c)})
fk = pd.DataFrame(rows)
fk.sort_values(["column", "parent (PK)"]) if len(fk) else print("No tables loaded yet.")

,parent (PK),child (FK),column,orphans,unused
9,SRC04/Category_Summary,SRC04/Products,category,0,0
0,SRC03/data,SRC01/data,customer_id,0,198
1,SRC03/data,SRC08/Sheet1,customer_id,0,157
2,SRC03/data,SRC09/data,customer_id,0,1033
52,SRC08/Sheet1,SRC01/data,customer_id,0,41
53,SRC08/Sheet1,SRC03/data,customer_id,157,0
54,SRC08/Sheet1,SRC09/data,customer_id,0,876
24,SRC07/Employee_v1_StartYear,SRC07/Employee_v2_Adjustment,date_of_birth,0,0
37,SRC07/Employee_v2_Adjustment,SRC07/Employee_v1_StartYear,date_of_birth,0,0
10,SRC06/data,SRC05/Orders,distributor_id,0,13


In [48]:
def check_fk(child_df, child_cols, parent_df, parent_cols):
    """Manual FK check; supports multiple columns and different names."""
    c = child_df[child_cols].drop_duplicates()
    p = parent_df[parent_cols].drop_duplicates()
    p.columns = child_cols
    m = c.merge(p, how="left", indicator=True)
    print("orphans:", (m["_merge"] == "left_only").sum(), "/", len(c))

# TODO: example
# check_fk(TABLES["SRC09"]["data"], ["original_order_id", "product_id"],
#          TABLES["SRC01"]["data"], ["order_id", "product_id"])